In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from catboost import Pool
import pygeohash as pgh
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from math import radians, cos, sin, asin, sqrt
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

In [2]:
BASE_DIR = Path("dataset")

TRAIN_PATH = BASE_DIR / "train.csv"
TEST_PATH  = BASE_DIR / "test.csv"
TRUE_VALUES_PATH = BASE_DIR / "y_true.csv"

In [3]:
# ─────────────────────────────────────────────────────────────
# 1. SPATIAL MATH & UTILITIES
# ─────────────────────────────────────────────────────────────

def build_neighbor_map(all_geohashes, loc_demand_map):
    """Uses pygeohash to shift lat/lon and find 4 immediate neighbors."""
    DELTA = 0.0055
    gh_set = set(all_geohashes)
    neighbor_means = {}
    
    for gh in all_geohashes:
        lat, lon = pgh.decode(gh)
        neighbors = []
        # Shift North, South, East, West
        for dlat, dlon in [(DELTA,0), (-DELTA,0), (0,DELTA), (0,-DELTA)]:
            candidate = pgh.encode(lat+dlat, lon+dlon, precision=6)
            if candidate in gh_set and candidate != gh:
                neighbors.append(candidate)
                
        if neighbors:
            vals = [loc_demand_map.get(n, np.nan) for n in neighbors]
            vals = [v for v in vals if not np.isnan(v)]
            neighbor_means[gh] = np.mean(vals) if vals else np.nan
        else:
            neighbor_means[gh] = np.nan
            
    return neighbor_means

def haversine(lon1, lat1, lon2, lat2):
    """Calculate the great circle distance in kilometers."""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 
    return c * r

In [4]:
# ─────────────────────────────────────────────────────────────
# 2. LOAD & MERGE
# ─────────────────────────────────────────────────────────────
print("Loading data...")
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

df_train['is_train'] = 1
df_test['is_train']  = 0
df_test['demand']    = np.nan

df_full = pd.concat([df_train, df_test], ignore_index=True)

Loading data...


In [5]:
# ─────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("Engineering features...")

df_full['RoadType'] = df_full['RoadType'].fillna('Unknown')
df_train['RoadType'] = df_train['RoadType'].fillna('Unknown')

# A. Cyclical Time
df_full['hour']   = df_full['timestamp'].str.split(':').str[0].astype(int)
df_full['minute'] = df_full['timestamp'].str.split(':').str[1].astype(int)
mins_of_day       = df_full['hour'] * 60 + df_full['minute']

df_full['time_sin'] = np.sin(2 * np.pi * mins_of_day / 1440)
df_full['time_cos'] = np.cos(2 * np.pi * mins_of_day / 1440)
df_full['time_bucket'] = mins_of_day // 15

df_full['is_rush_hour'] = (
    ((df_full['hour'] >= 7) & (df_full['hour'] < 9)) |
    ((df_full['hour'] >= 17) & (df_full['hour'] < 19))
).astype(int)

df_full['is_peak_window'] = (
    (df_full['hour'] >= 6) & (df_full['hour'] < 20)
).astype(int)

Engineering features...


In [6]:
def convert_time_to_minutes(t):
    h, m = map(int, t.split(':'))
    return h * 60 + m

df_full['time_minutes'] = df_full['timestamp'].apply(convert_time_to_minutes)

df_full['absolute_time'] = (
    (df_full['day'] - 48) * 1440
    + df_full['time_minutes']
)

In [7]:
df_full = df_full.sort_values(
    ['geohash', 'absolute_time']
).reset_index(drop=True)

In [8]:
# ============================================================
# CAUSAL TEMPORAL FEATURES (NO LEAKAGE)
# ============================================================

print("Creating causal lag features...")

# ------------------------------------------------------------
# 1. DEMAND YESTERDAY
# ------------------------------------------------------------

lookup_dict = df_train.set_index(
    ['geohash', 'timestamp', 'day']
)['demand'].to_dict()

df_full['demand_yesterday'] = df_full.apply(
    lambda row: lookup_dict.get(
        (row['geohash'], row['timestamp'], row['day'] - 1),
        np.nan
    ),
    axis=1
)

# B. Geo Features Extraction (Using PyGeohash)
print("Extracting coordinates via PyGeohash...")
df_full['lat'] = df_full['geohash'].apply(lambda x: pgh.decode(x)[0])
df_full['lon'] = df_full['geohash'].apply(lambda x: pgh.decode(x)[1])

# Distance to Center
busiest_geohash = df_train.groupby('geohash')['demand'].mean().idxmax()
center_lat, center_lon = pgh.decode(busiest_geohash)
df_full['dist_to_center'] = df_full.apply(
    lambda row: haversine(row['lon'], row['lat'], center_lon, center_lat), axis=1
)

# C. NEW: Spatial Clustering (Making something out of lat/lon)
print("Creating Spatial Zones using K-Means...")
kmeans = KMeans(n_clusters=20, random_state=42)
df_full['zone_id'] = kmeans.fit_predict(df_full[['lat', 'lon']])
df_full['zone_id'] = df_full['zone_id'].astype('category')

Creating causal lag features...
Extracting coordinates via PyGeohash...
Creating Spatial Zones using K-Means...


In [9]:
# ============================================================
# CATEGORICALS + CLEANUP
# ============================================================

df_full['zone_road'] = (
    df_full['zone_id'].astype(str)
    + '_'
    + df_full['RoadType'].astype(str)
).astype('category')

df_full['RoadType'] = (
    df_full['RoadType']
    .fillna('Unknown')
    .astype('category')
)

df_full['Weather'] = (
    df_full['Weather']
    .fillna('Unknown')
    .astype('category')
)

df_full['LargeVehicles'] = (
    df_full['LargeVehicles']
    .fillna('Unknown')
    .astype('category')
)

df_full['Landmarks'] = (
    df_full['Landmarks']
    .fillna('Unknown')
    .astype('category')
)

df_full['NumberofLanes'] = (
    df_full['NumberofLanes']
    .fillna(df_full['NumberofLanes'].median())
)

df_full['Temperature'] = (
    df_full['Temperature']
    .fillna(df_full['Temperature'].median())
)

# df_full['temp_bin'] = pd.cut(
#     df_full['Temperature'],
#     bins=[-np.inf, 5, 15, 25, 35, np.inf],
#     labels=[0, 1, 2, 3, 4]
# ).astype(int)

In [10]:
# ─────────────────────────────────────────────────────────────
# 4. SPLIT & LOG TRANSFORM
# ─────────────────────────────────────────────────────────────
train_df = df_full[(df_full['is_train'] == 1) & (df_full['demand'].notnull())].copy()

val_times = ['1:30', '1:45', '2:0']

val_mask = (
    (train_df['day'] == 49) &
    (train_df['timestamp'].isin(val_times))
)

train_mask = ~val_mask

test_df  = df_full[df_full['is_train'] == 0].copy()

In [11]:
# ============================================================
# SAFE GLOBAL AGGREGATES (NO VALIDATION LEAKAGE)
# ============================================================

safe_train = train_df.loc[train_mask].copy()

# ------------------------------------------------------------
# LOCATION STATS
# ------------------------------------------------------------

loc_stats = safe_train.groupby('geohash')['demand'].agg([
    ('loc_mean', 'mean'),
    ('loc_std', 'std'),
    ('loc_median', 'median'),
    ('loc_max', 'max')
]).reset_index()

train_df = train_df.merge(loc_stats, on='geohash', how='left')
test_df  = test_df.merge(loc_stats, on='geohash', how='left')

# ------------------------------------------------------------
# FILL SAFE
# ------------------------------------------------------------

agg_cols = [
    'loc_mean',
    'loc_std',
    'loc_median',
    'loc_max',
]

for col in agg_cols:
    fill_val = safe_train['demand'].mean()
    
    train_df[col] = train_df[col].fillna(fill_val)
    test_df[col]  = test_df[col].fillna(fill_val)

In [12]:
# ============================================================
# REBUILD TEMPORAL MASKS AFTER MERGES
# ============================================================

val_mask = (
    (train_df['day'] == 49) &
    (train_df['timestamp'].isin(val_times))
)

train_mask = ~val_mask

FEATURES = [
    'time_sin',
    'time_cos',
    'time_bucket',

    'lat',
    'lon',
    'dist_to_center',
    'zone_id',
    "geohash",

    'demand_yesterday',

    'loc_mean',
    'loc_std',
    'loc_median',
    'loc_max',

    # 'zone_road',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    # 'Weather',
    'Temperature',
    # 'temp_bin',
]

In [15]:
print("\nStarting Temporal Validation Training...")

# --------------------------------------------
# TRAIN / VALID SPLIT
# --------------------------------------------

X_train = train_df.loc[train_mask, FEATURES].copy()
y_train = np.log1p(train_df.loc[train_mask, 'demand'])

X_valid = train_df.loc[val_mask, FEATURES].copy()
y_valid = np.log1p(train_df.loc[val_mask, 'demand'])

X_test_final = test_df[FEATURES].copy()

# --------------------------------------------
# TARGET ENCODING
# --------------------------------------------

SMOOTH = 10

global_mean = y_train.mean()

enc_df = pd.DataFrame({
    'geohash': train_df.loc[train_mask, 'geohash'],
    'target': y_train.values
})

gh_stats = enc_df.groupby('geohash')['target'].agg(['mean', 'count'])

gh_stats['encoded'] = (
    (gh_stats['mean'] * gh_stats['count'] + global_mean * SMOOTH)
    / (gh_stats['count'] + SMOOTH)
)

gh_enc_map = gh_stats['encoded'].to_dict()

feat_cols = FEATURES

cat_features = [
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    # 'Weather',
    'zone_id',
    # 'zone_road',
    'geohash'
]

# ============================================================
# CATBOOST POOLS
# ============================================================

train_pool = Pool(
    data=X_train[feat_cols],
    label=y_train,
    cat_features=cat_features
)

valid_pool = Pool(
    data=X_valid[feat_cols],
    label=y_valid,
    cat_features=cat_features
)

test_pool = Pool(
    data=X_test_final[feat_cols],
    cat_features=cat_features
)

# ============================================================
# CATBOOST MODEL
# ============================================================

model = CatBoostRegressor(
    loss_function='RMSE',

    iterations=5000,
    learning_rate=0.02,
    depth=6,

    l2_leaf_reg=5,

    random_seed=42,

    eval_metric='RMSE',

    bootstrap_type='Bayesian',
    bagging_temperature=1,

    verbose=100
)

model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True,
    early_stopping_rounds=200
)

# --------------------------------------------
# VALIDATION SCORE
# --------------------------------------------

valid_preds = np.expm1(
    model.predict(valid_pool)
)

r2 = r2_score(
    np.expm1(y_valid),
    valid_preds
)

print(f"\nTemporal Validation R²: {r2:.6f}")

# --------------------------------------------
# TEST PREDICTION
# --------------------------------------------

test_preds = np.expm1(
    model.predict(test_pool)
)


Starting Temporal Validation Training...
0:	learn: 0.1069444	test: 0.1103098	best: 0.1103098 (0)	total: 66.5ms	remaining: 5m 32s
100:	learn: 0.0374447	test: 0.0431836	best: 0.0431836 (100)	total: 785ms	remaining: 38.1s
200:	learn: 0.0313995	test: 0.0368141	best: 0.0368141 (200)	total: 1.56s	remaining: 37.2s
300:	learn: 0.0297294	test: 0.0342992	best: 0.0342970 (299)	total: 2.24s	remaining: 35s
400:	learn: 0.0287929	test: 0.0329147	best: 0.0329147 (400)	total: 2.94s	remaining: 33.8s
500:	learn: 0.0281325	test: 0.0320757	best: 0.0320757 (500)	total: 3.67s	remaining: 33s
600:	learn: 0.0275677	test: 0.0313279	best: 0.0313279 (600)	total: 4.4s	remaining: 32.2s
700:	learn: 0.0270939	test: 0.0308136	best: 0.0308136 (700)	total: 5.14s	remaining: 31.5s
800:	learn: 0.0266915	test: 0.0304672	best: 0.0304672 (800)	total: 5.88s	remaining: 30.8s
900:	learn: 0.0263469	test: 0.0301018	best: 0.0301018 (900)	total: 6.62s	remaining: 30.1s
1000:	learn: 0.0260549	test: 0.0297161	best: 0.0297161 (1000)	tot

In [16]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({
    'feature': feat_cols,
    'importance': model.get_feature_importance()
})

importance_df = importance_df.sort_values(
    'importance',
    ascending=False
)

print("\nTop Feature Importances:")
print(importance_df.to_string(index=False))


Top Feature Importances:
         feature  importance
        RoadType   36.307971
        loc_mean   12.098951
      loc_median   10.665633
        time_sin    5.608003
   LargeVehicles    5.546803
        time_cos    5.076104
     time_bucket    4.979403
         loc_std    4.306821
         loc_max    4.232996
   NumberofLanes    3.782363
         geohash    1.522653
demand_yesterday    1.399944
         zone_id    1.330464
             lon    1.261931
             lat    0.888494
  dist_to_center    0.761793
     Temperature    0.194866
       Landmarks    0.034806


In [17]:
# ─────────────────────────────────────────────────────────────
# 6. STRICT SUBMISSION
# ─────────────────────────────────────────────────────────────
print("\nCreating submission...")
test_preds = np.maximum(test_preds, 0)

submission = pd.read_csv(TEST_PATH)[['Index']].copy()
pred_map   = dict(zip(test_df['Index'].values, test_preds))
submission['demand'] = submission['Index'].map(pred_map)

missing = submission['demand'].isna().sum()
if missing > 0:
    print(f"Warning: Filling {missing} missing rows with global train mean.")
    submission['demand'] = submission['demand'].fillna(np.expm1(y).mean())


Creating submission...


In [18]:
# ─────────────────────────────────────────────────────────────
# 7. Compare with True Values
# ─────────────────────────────────────────────────────────────
true_values_df = pd.read_csv(TRUE_VALUES_PATH)

y_true = true_values_df['demand']
y_pred = submission['demand']

r2 = r2_score(y_true, y_pred)

print("\nEvaluation Metrics")
print(f"R²: {r2:.6f}")


Evaluation Metrics
R²: 0.906963


In [19]:
submission.to_csv(f"submissions/submission_final_{r2*100:.6f}.csv", index=False)
print(f"Saved submissions/submission_final_{r2*100:.6f}.csv ({len(submission)} rows)")

Saved submissions/submission_final_90.696329.csv (41778 rows)


In [20]:
# import plotly.express as px
# import pygeohash as pgh

# # 1. Calculate the average demand for every unique geohash in your training data
# geo_map_df = df_train.groupby('geohash')['demand'].mean().reset_index()

# # 2. Decode the coordinates using the pygeohash library
# geo_map_df['lat'] = geo_map_df['geohash'].apply(lambda x: pgh.decode(x)[0])
# geo_map_df['lon'] = geo_map_df['geohash'].apply(lambda x: pgh.decode(x)[1])

# # 3. Render the interactive map
# fig = px.scatter_mapbox(
#     geo_map_df, 
#     lat="lat", 
#     lon="lon", 
#     color="demand",                         # Heatmap coloring based on busyness
#     color_continuous_scale="Plasma", 
#     size="demand",                          # Bubbles get larger with higher demand
#     zoom=10, 
#     mapbox_style="carto-positron",          # Uses a clean, street-level base map
#     title="Average Traffic Demand by Geohash"
# )

# # 4. Increase the size of the plot for better visibility
# fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0}, height=700)
# fig.show(renderer="notebook")